# Tema 1 — Explorarea corpusului și primul prompt
În acest notebook vei explora corpusul curățat de comentarii YouTube și vei testa un prim prompt exploratoriu.

Vei testa 10 comentarii și vei reflecta asupra unor probleme precum ambiguitatea, țintele multiple, sarcasmul și confuzia dintre sentiment și poziționarea față de țintă.

## 1. Pregătire
Încărcăm bibliotecile necesare și cheia API pentru Gemini.
Modificați doar celula de configurare a studentului.

In [17]:
from pathlib import Path
import os
import sys
import json
import random
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

In [18]:
ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
print("Root project:", ROOT)
print("Gemini key found:", GEMINI_API_KEY is not None)

Root project: c:\Users\Carmen Ciutu\Desktop\proiect AI\echochamber-project-team-4
Gemini key found: True


## 2. Configurare
Modificați  această celulă.
Schimbați `student_id` cu folderul vostru: `student_01`, `student_02`, etc.

In [39]:
student_id = "student 4"
model = "gemini-2.5-flash-lite"
temperature = 0.2
corpus_file = ROOT / "data" / "cleaned" / "corpus_youtube_large_clean.jsonl"
output_file = ROOT / "outputs" / f"{student_id}_prompt_outputs.csv"

## 3. Încărcăm corpusul curățat
Corpusul este salvat în format JSONL.
JSONL înseamnă: un comentariu pe fiecare linie.

In [20]:
# Citim fiecare linie din fișierul JSONL si o transformăm într-un dataframe pentru explorare
records = []
with corpus_file.open("r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))
# Transformăm lista într-un DataFrame pentru explorare mai ușoară
df = pd.DataFrame(records)
df.head()

,id,source_platform,source_channel,text,text_raw,bubble_label,bubble_self_identified,topic,rhetoric_type,video_id,video_title,video_date,comment_date,likes,lang,collected_at
0,yt_5rHoTX3U_3Q_UgxaV5so7vyeXpyy8up4AaABAg,youtube,georgesimionoficial,Felicitării George Simion Președintele Românie...,Felicitării George Simion Președintele Românie...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-23,1,ro,2026-03-22
1,yt_5rHoTX3U_3Q_UgwJYiRLMbLfl2AipVR4AaABAg,youtube,georgesimionoficial,Asa trebuie să fiți printre oameni nu sa se do...,Asa trebuie să fiți printre oameni nu sa se do...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-02,5,ro,2026-03-22
2,yt_5rHoTX3U_3Q_UgzXqOk_SypZcQS-JcF4AaABAg,youtube,georgesimionoficial,Eu am votat cu George Simion din primul tur pt...,Eu am votat cu George Simion din primul tur pt...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-01,30,ro,2026-03-22
3,yt_5rHoTX3U_3Q_UgzpKghDX0_l-Gc3P4V4AaABAg,youtube,georgesimionoficial,Si de trebuie deposite de combustibil degeaba ...,Si de trebuie deposite de combustibil degeaba ...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-16,3,ro,2026-03-22
4,yt_5rHoTX3U_3Q_UgwqOTRSHNPj9cuGwNt4AaABAg,youtube,georgesimionoficial,Nu te descuraja că dobitoci și proști vor fi p...,Nu te descuraja că dobitoci și proști vor fi...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-01,9,ro,2026-03-22


In [21]:
print("Number of comments:", len(df))
print("Columns:", list(df.columns))

Number of comments: 30451
Columns: ['id', 'source_platform', 'source_channel', 'text', 'text_raw', 'bubble_label', 'bubble_self_identified', 'topic', 'rhetoric_type', 'video_id', 'video_title', 'video_date', 'comment_date', 'likes', 'lang', 'collected_at']


## 4. Explorare rapidă a corpusului
Ne uităm la canalele principale și la câteva exemple de comentarii.
Această etapă ne ajută să înțelegem ce tip de date avem înainte să folosim modelul.

In [22]:
# cele mai frecvente 15 canale sursă din dataset
df["source_channel"].value_counts().head(15) # completează pentru a vedea cele mai frecvente 15 canale sursă din dataset

source_channel
RecorderRomania                   12177
turcescu111                        5019
georgesimionoficial                3669
CălinGeorgescu-CanalulOficial      3460
@CălinGeorgescu-CanalulOficial     2557
TuDecizi-s3g                        647
StareaNatiei                        623
AltcevacuAdrianArtene               363
roxindaniel                         305
otvdirect                           304
digi24hd56                          265
euronewsro                          238
DianaSosoacaOfficial                227
AdevaruriSecrete                    180
g4media479                          158
Name: count, dtype: int64

In [23]:
# aruncă o privire asupra unor comentarii random din dataset
df[["source_channel", "video_title", "text"]].sample(5, random_state=42)

,source_channel,video_title,text
23002,CălinGeorgescu-CanalulOficial,Călin Georgescu - Pacea de la București ( IPJ ...,Multă sănătate dl. Președinte Călin Georgescu....
5815,@CălinGeorgescu-CanalulOficial,Călin Georgescu - De ce vorbim despre Eminescu...,Un discurs care trebuia sa vina de la Cotrocen...
11191,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,Autoritatile abilitate sa intervina!!! De acee...
11316,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,In acest moment mai putem spune doar Dumnezeu ...
12505,RecorderRomania,DOCUMENTAR RECORDER. Singuri,E dureros.. e crunt.. simt vinovatie si recuno...


## 5. Alegem 10 comentarii pentru testarea promptului
Folosim 10 comentarii curate.
Puteți păstra eșantionarea aleatorie sau puteți selecta manual comentarii mai interesante.

In [24]:
sample_df = df.sample(10, random_state=42).copy()
sample_df[["source_channel", "text"]]

,source_channel,text
23002,CălinGeorgescu-CanalulOficial,Multă sănătate dl. Președinte Călin Georgescu....
5815,@CălinGeorgescu-CanalulOficial,Un discurs care trebuia sa vina de la Cotrocen...
11191,RecorderRomania,Autoritatile abilitate sa intervina!!! De acee...
11316,RecorderRomania,In acest moment mai putem spune doar Dumnezeu ...
12505,RecorderRomania,E dureros.. e crunt.. simt vinovatie si recuno...
9644,RecorderRomania,Cite dosare ați judecat și nu ați recuperat ni...
23843,turcescu111,"Totul duce către: Noua Ordine Mondială, pentru..."
11605,RecorderRomania,"Un hot corupt arogant si nesimtit, caruia nime..."
15486,RecorderRomania,"4:30 și încă 1% rămas pentru Crin Alcoolescu, ..."
7767,RecorderRomania,Vă mai dau niște firme din Galați care au alți...


Optional , poti alege sa folosesti  alta metoda de esantionare sau sa filtrezi dupa anumite canale sursa sau alte criterii. Important e sa ai un set de date mic pe care sa testezi promptul inainte de a-l rula pe intregul dataset.

## 6. Primul prompt exploratoriu
Completăm un prompt simplu pentru analizarea comentariilor politice.
Promptul trebuie să ceară:
- ținta comentariului;
- poziționarea față de țintă;
- tonul;
- tema;
- problema de interpretare;
- o justificare scurtă.
Important: tonul sau sentimentul general nu este același lucru cu poziționarea față de țintă.

In [25]:
# IMPORTANT - SCHIMBA PROMTUL DE MAI JOS PENTRU A SE POTRIVI CU CERINȚELE TALE ȘI ASIGURĂ-TE CĂ RESPECTĂ STRUCTURA SOLICITATĂ
# include în prompt instrucțiuni clare pentru fiecare dintre cele 7 elemente pe care vrei să le extragi și asigură-te că modelul înțelege că trebuie să returneze un JSON valid cu exact acele chei
# inlocueste "..." cu instrucțiuni clare pentru fiecare element
# Prompt de sistem: definește rolul modelului
# poti pune si alte axe de analiza care te intereseaza


SYSTEM_PROMPT = """
Ești un analist expert pentru comentarii politice în limba română.
Analizezi fiecare comentariu cu atenție și extragi doar informații care pot fi susținute de text.
Pentru comentariile conspiraționiste, fii atent la suspiciuni față de instituții, elite ascunse, actori externi, manipulare globală, agende secrete și explicații totalizante.
Nu inventa informații și nu adăuga explicații în afara textului.
Returnează exclusiv un JSON valid, fără markdown, fără explicații și fără text suplimentar.
"""

USER_PROMPT_TEMPLATE = """
Citește următorul comentariu politic și identifică:
1. target: cine sau ce este vizat direct de comentariu
2. stance: poziționarea față de target; pentru acest tip de comentariu poate fi suspicios, acuzator, ironic sau denunțător
3. sentiment: sentimentul general exprimat în comentariu
4. tone: tonul discursiv; pentru această bulă poate fi alarmist, hiper-suspicios, speculativ, revelator sau dramatic
5. topic: tema principală discutată
6. interpretation_problem: ce este ambiguu, exagerat sau dificil de interpretat în comentariu
7. reason: justificare scurtă, de 1-2 propoziții, bazată strict pe text

Important:
- Fii atent la semnele de conspiraționism: forțe ascunse, elite globale, manipulare, control extern, agenda globală, adevăruri ascunse.
- Distinge între sentimentul general și poziția față de target.
- Dacă un element nu este clar, folosește o valoare prudentă precum "unknown" sau "unclear".
- Returnează JSON valid cu exact aceste chei:
  target, stance, sentiment, tone, topic, interpretation_problem, reason
- Nu adăuga niciun alt câmp.
- Nu scrie explicații, doar JSON.

Comentariu:
<<< {comment_text} >>>
"""

## 7. Conectarea la model
Folosim Gemini prin endpoint compatibil cu OpenAI.
Modelul și temperatura au fost setate mai sus.

In [29]:
from openai import OpenAI

assert GEMINI_API_KEY, "GEMINI_API_KEY nu este setată în .env"

client = OpenAI(
    api_key=GEMINI_API_KEY,
    base_url=BASE_URL,
)

In [30]:
def annotate_comment(comment_text):
    prompt = USER_PROMPT_TEMPLATE.format(comment_text=comment_text)
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

## 8. Rulăm promptul pe 10 comentarii
Trimitem fiecare comentariu selectat la model și salvăm răspunsurile.

In [31]:
n_comments = 10  # schimbă aici: 3, 5 sau 10
sample_for_prompt = df.head(n_comments)

outputs = []
for _, row in sample_for_prompt.iterrows():
    outputs.append({
        "source_channel": row.get("source_channel", ""),
        "video_title": row.get("video_title", ""),
        "comment_text": row["text"],
        "model_output": annotate_comment(row["text"])
    })
results_df = pd.DataFrame(outputs)
results_df

,source_channel,video_title,comment_text,model_output
0,georgesimionoficial,Turul României: realități de la firul ierbii,Felicitării George Simion Președintele Românie...,"```json\n{\n ""target"": ""George Simion"",\n ""s..."
1,georgesimionoficial,Turul României: realități de la firul ierbii,Asa trebuie să fiți printre oameni nu sa se do...,"```json\n{\n ""target"": ""George Simion și parl..."
2,georgesimionoficial,Turul României: realități de la firul ierbii,Eu am votat cu George Simion din primul tur pt...,"```json\n{\n ""target"": ""George Simion"",\n ""s..."
3,georgesimionoficial,Turul României: realități de la firul ierbii,Si de trebuie deposite de combustibil degeaba ...,"```json\n{\n ""target"": ""Guvernul/Autoritățile..."
4,georgesimionoficial,Turul României: realități de la firul ierbii,Nu te descuraja că dobitoci și proști vor fi p...,"```json\n{\n ""target"": ""Guvernul României, Ni..."
5,georgesimionoficial,Turul României: realități de la firul ierbii,"Salutare Simioane, bine că te freacă la creier...","```json\n{\n ""target"": ""George Simion, Klaus ..."
6,georgesimionoficial,Turul României: realități de la firul ierbii,Vă felicit domnule George că mergeți pe teren ...,"```json\n{\n ""target"": ""George"",\n ""stance"":..."
7,georgesimionoficial,Turul României: realități de la firul ierbii,Tot aud că Simion e un monstru politic. Așa sp...,"```json\n{\n ""target"": ""George Simion"",\n ""s..."
8,georgesimionoficial,Turul României: realități de la firul ierbii,"Așa da, domnule Simion! Tot înainte, până la v...","```json\n{\n ""target"": ""George Simion"",\n ""s..."
9,georgesimionoficial,Turul României: realități de la firul ierbii,Respect George pentru sacrificiile pe care le ...,"```json\n{\n ""target"": ""George și echipa sa p..."


# 9. Verificam rezultatele

In [34]:
results_df.model_output[3]

'```json\n{\n  "target": "Guvernul/Autoritățile responsabile cu aprovizionarea cu combustibil",\n  "stance": "Acuzator",\n  "sentiment": "Nervos",\n  "tone": "Alarmist",\n  "topic": "Aprovizionarea cu combustibil și stocurile strategice",\n  "interpretation_problem": "Nu este clar cine ar trebui să caute sursele de petrol și de ce stocurile actuale sunt considerate insuficiente.",\n  "reason": "Comentariul critică lipsa de acțiune a autorităților în ceea ce privește stocurile de combustibil, sugerând că resursele militare (tancuri, blindate) sunt inutile dacă nu există combustibil. Se pune accent pe necesitatea de a căuta și cumpăra cât mai mult combustibil."\n}\n```'

In [33]:
# funcție pentru a curăța și parsa output-ul modelului, care poate conține JSON în diferite formate (text simplu sau bloc ```json)

def parse_model_output(text):
    # Modelul poate întoarce JSON ca text simplu sau în bloc ```json
    text = text.replace("```json", "")
    text = text.replace("```", "")
    text = text.strip()
    
    return json.loads(text)

In [35]:
parsed_outputs = []

for _, row in results_df.iterrows():
    parsed = parse_model_output(row["model_output"])
    
    parsed_outputs.append({
        "source_channel": row["source_channel"],
        "video_title": row["video_title"],
        "comment_text": row["comment_text"],
        "target": parsed.get("target", ""),
        "stance": parsed.get("stance", ""),
        "sentiment": parsed.get("sentiment", ""),
        "tone": parsed.get("tone", ""),
        "topic": parsed.get("topic", ""),
        "interpretation_problem": parsed.get("interpretation_problem", ""),
        "reason": parsed.get("reason", "")
    })

parsed_df = pd.DataFrame(parsed_outputs)
parsed_df

,source_channel,video_title,comment_text,target,stance,sentiment,tone,topic,interpretation_problem,reason
0,georgesimionoficial,Turul României: realități de la firul ierbii,Felicitării George Simion Președintele Românie...,George Simion,acuzator,negativ,speculativ,politică,Comentariul folosește un limbaj excesiv de poz...,"Comentariul pare să fie ironic, folosind un li..."
1,georgesimionoficial,Turul României: realități de la firul ierbii,Asa trebuie să fiți printre oameni nu sa se do...,George Simion și parlamentarii,susținător,pozitiv,aprobator,activitatea parlamentară și votul cetățenilor,unclear,Comentariul laudă activitatea domnului Simion ...
2,georgesimionoficial,Turul României: realități de la firul ierbii,Eu am votat cu George Simion din primul tur pt...,George Simion,susținător,speranță,optimist,alegeri prezidențiale,unclear,Comentariul exprimă susținere pentru George Si...
3,georgesimionoficial,Turul României: realități de la firul ierbii,Si de trebuie deposite de combustibil degeaba ...,Guvernul/Autoritățile responsabile cu aprovizi...,Acuzator,Nervos,Alarmist,Aprovizionarea cu combustibil și stocurile str...,Nu este clar cine ar trebui să caute sursele d...,Comentariul critică lipsa de acțiune a autorit...
4,georgesimionoficial,Turul României: realități de la firul ierbii,Nu te descuraja că dobitoci și proști vor fi p...,"Guvernul României, Nicușor Dan, Bolojan, Predo...","acuzator, denunțător",negativ,"alarmist, dramatic",Critica politică și susținerea AUR,Lista de nume pare să fie o colecție de politi...,"Comentariul critică ""guvernul României"" și lis..."
5,georgesimionoficial,Turul României: realități de la firul ierbii,"Salutare Simioane, bine că te freacă la creier...","George Simion, Klaus Iohannis, Dominic Fritz","acuzator, ironic",negativ,ironic,critica activității politice și a deplasărilor...,expresia 'te freacă la creier' este o metaforă...,Comentariul critică activitatea lui George Sim...
6,georgesimionoficial,Turul României: realități de la firul ierbii,Vă felicit domnule George că mergeți pe teren ...,George,apreciativ,pozitiv,recunoscător,implicarea în problemele oamenilor,unknown,Comentariul exprimă felicitări și apreciere pe...
7,georgesimionoficial,Turul României: realități de la firul ierbii,Tot aud că Simion e un monstru politic. Așa sp...,George Simion,acuzator,negativ,speculativ,Percepția publică a lui George Simion,Comparația lui George Simion cu Vlad Țepeș est...,Comentariul compară direct pe George Simion cu...
8,georgesimionoficial,Turul României: realități de la firul ierbii,"Așa da, domnule Simion! Tot înainte, până la v...",George Simion,acuzator,pozitiv,entuziast,susținere politică,unknown,Comentariul exprimă susținere și încurajare di...
9,georgesimionoficial,Turul României: realități de la firul ierbii,Respect George pentru sacrificiile pe care le ...,George și echipa sa politică,susținător,pozitiv,încurajator,ascensiunea politică și reformarea țării,unclear,Comentariul exprimă admirație și încredere în ...


# 10 Salvarea csv si inspectarea rezulatelor
- salvati ca csv 
- deschideti csv si verificati rezultatele 
- raspundeti la urmatoarele intrebare: promptul separă corect sentimentul general de poziționarea față de target? 

In [40]:
output_file.parent.mkdir(parents=True, exist_ok=True)
parsed_df.to_csv(output_file, index=False, encoding="utf-8-sig", sep=";")
print(f"Rezultatele au fost salvate în: {output_file}")
parsed_df.head(10)

Rezultatele au fost salvate în: c:\Users\Carmen Ciutu\Desktop\proiect AI\echochamber-project-team-4\outputs\student 4_prompt_outputs.csv


,source_channel,video_title,comment_text,target,stance,sentiment,tone,topic,interpretation_problem,reason
0,georgesimionoficial,Turul României: realități de la firul ierbii,Felicitării George Simion Președintele Românie...,George Simion,acuzator,negativ,speculativ,politică,Comentariul folosește un limbaj excesiv de poz...,"Comentariul pare să fie ironic, folosind un li..."
1,georgesimionoficial,Turul României: realități de la firul ierbii,Asa trebuie să fiți printre oameni nu sa se do...,George Simion și parlamentarii,susținător,pozitiv,aprobator,activitatea parlamentară și votul cetățenilor,unclear,Comentariul laudă activitatea domnului Simion ...
2,georgesimionoficial,Turul României: realități de la firul ierbii,Eu am votat cu George Simion din primul tur pt...,George Simion,susținător,speranță,optimist,alegeri prezidențiale,unclear,Comentariul exprimă susținere pentru George Si...
3,georgesimionoficial,Turul României: realități de la firul ierbii,Si de trebuie deposite de combustibil degeaba ...,Guvernul/Autoritățile responsabile cu aprovizi...,Acuzator,Nervos,Alarmist,Aprovizionarea cu combustibil și stocurile str...,Nu este clar cine ar trebui să caute sursele d...,Comentariul critică lipsa de acțiune a autorit...
4,georgesimionoficial,Turul României: realități de la firul ierbii,Nu te descuraja că dobitoci și proști vor fi p...,"Guvernul României, Nicușor Dan, Bolojan, Predo...","acuzator, denunțător",negativ,"alarmist, dramatic",Critica politică și susținerea AUR,Lista de nume pare să fie o colecție de politi...,"Comentariul critică ""guvernul României"" și lis..."
5,georgesimionoficial,Turul României: realități de la firul ierbii,"Salutare Simioane, bine că te freacă la creier...","George Simion, Klaus Iohannis, Dominic Fritz","acuzator, ironic",negativ,ironic,critica activității politice și a deplasărilor...,expresia 'te freacă la creier' este o metaforă...,Comentariul critică activitatea lui George Sim...
6,georgesimionoficial,Turul României: realități de la firul ierbii,Vă felicit domnule George că mergeți pe teren ...,George,apreciativ,pozitiv,recunoscător,implicarea în problemele oamenilor,unknown,Comentariul exprimă felicitări și apreciere pe...
7,georgesimionoficial,Turul României: realități de la firul ierbii,Tot aud că Simion e un monstru politic. Așa sp...,George Simion,acuzator,negativ,speculativ,Percepția publică a lui George Simion,Comparația lui George Simion cu Vlad Țepeș est...,Comentariul compară direct pe George Simion cu...
8,georgesimionoficial,Turul României: realități de la firul ierbii,"Așa da, domnule Simion! Tot înainte, până la v...",George Simion,acuzator,pozitiv,entuziast,susținere politică,unknown,Comentariul exprimă susținere și încurajare di...
9,georgesimionoficial,Turul României: realități de la firul ierbii,Respect George pentru sacrificiile pe care le ...,George și echipa sa politică,susținător,pozitiv,încurajator,ascensiunea politică și reformarea țării,unclear,Comentariul exprimă admirație și încredere în ...


### Interpretare
Per total, rezultatele sunt parțial corecte: comentariile de susținere au fost etichetate bine în majoritatea cazurilor, iar comentariile critice au fost surprinse destul de clar. Totuși, nu toate textele sunt ironice; de exemplu, unele comentarii sunt doar laudative sau acuzatoare, iar modelul le-a marcat uneori prea repede ca fiind ironice sau speculativ-acuzatoare. În plus, s-a văzut că sentimentul general și poziționarea față de target se confundă uneori, mai ales la textele cu emoji-uri, exagerări sau formulări ambigue. Așadar, rezultatele sunt utile, dar ar trebui rafinate cu exemple mai clare și reguli mai stricte pentru detectarea ironiei și separarea dintre sentiment și stance.